In [ ]:
import ollama, time
def ollama_response(question:str):
    started = time.time()
    response = ollama.chat( model='gemma3:4b',  # Or 'tinyllama', 'smollm2:1.7b'
    messages=[{'role': 'user', 'content':question  }],
               
)
    elapsed = time.time() - started
    #print(f"time taken is {elapsed}")
    #print(response['message']['content'])
    print(response)
    return response

In [15]:

resp = ollama_response("Give a bief response about whether for every general question I ask that is factual, you can provide a citation ")

model='gemma3:4b' created_at='2026-09-02T08:06:57.9995683Z' done=True done_reason='stop' total_duration=16219082600 load_duration=4740041400 prompt_eval_count=33 prompt_eval_duration=827645000 eval_count=71 eval_duration=10647106000 message=Message(role='assistant', content="Yes, absolutely! I am designed to provide factual information and, whenever possible, I will include a citation to support my answer. My goal is to give you reliable information with verifiable sources. 😊 \n\nTo help me do this effectively, please feel free to let me know if you'd like me to prioritize citations for specific types of questions.", thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [ ]:
print(type(resp))
print(resp.message.content)
print(resp.prompt_eval_count)
print(resp.eval_count)

In [ ]:
ollama_client = ollama.AsyncClient()

In [ ]:
resp = ollama_response(
    "Give a brief response about whether for every general question I ask that is factual, you can provide a citation"
)
print("TYPE OUTSIDE FUNCTION:", type(resp))
print("RESPONSE OUTSIDE FUNCTION:", resp)
resp[]

TYPE INSIDE FUNCTION: <class 'ollama._types.ChatResponse'>
RESPONSE INSIDE FUNCTION: model='gemma3:4b' created_at='2026-09-02T08:11:23.5085536Z' done=True done_reason='stop' total_duration=7538398800 load_duration=3191900 prompt_eval_count=31 prompt_eval_duration=298706000 eval_count=50 eval_duration=7225379000 message=Message(role='assistant', content='Yes, absolutely! I will always strive to provide a citation for any factual answer I give you. My goal is to be as transparent and verifiable as possible. 😊 \n\nJust let me know if you’d like to see the source!', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
TYPE OUTSIDE FUNCTION: <class 'ollama._types.ChatResponse'>
RESPONSE OUTSIDE FUNCTION: model='gemma3:4b' created_at='2026-09-02T08:11:23.5085536Z' done=True done_reason='stop' total_duration=7538398800 load_duration=3191900 prompt_eval_count=31 prompt_eval_duration=298706000 eval_count=50 eval_duration=7225379000 message=Message(role='assistant', content

In [ ]:
print(resp.prompt_eval_count)
print(resp.eval_count)

31
50


## ** Async Ollama**


In [ ]:
import ollama, time
ollama_client = ollama.AsyncClient()
async def ollama_response(question:str):
    started = time.time()
    response = await ollama_client.chat( model='gemma3:4b',  # Or 'tinyllama', 'smollm2:1.7b'
    messages=[{'role': 'user', 'content':question  }],
        options={
        'temperature': 0.0},
               
)
    elapsed = time.time() - started
    print("TYPE INSIDE FUNCTION:", type(response))
    print("RESPONSE INSIDE FUNCTION:", response)

    #print(f"time taken is {elapsed}")
    #print(response['message']['content'])
    #print(response)
    return response

In [24]:
resp = await ollama_response("Give a brief response about whether for every general question I ask that is factual, you can provide a citation ")

TYPE INSIDE FUNCTION: <class 'ollama._types.ChatResponse'>
RESPONSE INSIDE FUNCTION: model='gemma3:4b' created_at='2026-09-02T08:31:20.401237Z' done=True done_reason='stop' total_duration=10478464600 load_duration=4536228400 prompt_eval_count=32 prompt_eval_duration=703154000 eval_count=38 eval_duration=5232487000 message=Message(role='assistant', content='Yes, absolutely! I’ll do my best to provide a citation for every factual answer I give you. My goal is to be as transparent and reliable as possible. 😊 \n', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [25]:
from openai import AsyncOpenAI
from dotenv import load_dotenv
import os
load_dotenv(r"C:\Users\prahn\OneDrive\Documents\IITM-Pravartak\Pravartak_Practice\practice_scripts\.env")
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

is_local=True
RATES: dict[str, tuple[float, float]] = {
    "gpt-4o-mini": (0.15, 0.60),
    "gpt-4o":      (2.50, 10.00),
}
def compute_cost_usd(rates:dict, prompt_tokens: int, completion_tokens: int) -> float:
    model="gpt-4o-mini"
    rates = RATES.get(model)
    if rates is None:
        return 0.0
    in_rate, out_rate = rates
    return round((prompt_tokens * in_rate + completion_tokens * out_rate) / 1_000_000.0, 4)

async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    snippet_ID = snippet['id']
    messages = STRATEGIES[strategy_name](snippet['snippet'])
    output = {}
    if is_local:
        ollama_client = ollama.AsyncClient()
        started = time.time()
        response = await ollama_client.chat( model='gemma3:4b',
        messages=messages,
        options={
        'temperature': 0.0},  
        )
        elapsed = round(time.time() - started, 3)
        raw_response = response['message']['content']
        output['strategy_name'] = strategy_name
        output['snippet_ID'] = snippet_ID
        output['raw_response'] = raw_response
        output['parsed_extraction'] = parse_response(raw_response)
        output['cost_USD'] = compute_cost_usd(RATES, prompt_tokens=response.prompt_eval_count, completion_tokens=response.eval_count)
        output['latency_s'] = elapsed
        
        return output
    else:
        started = time.time()

        response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.0,
        )

        elapsed = round(time.time() - started, 3)

        raw_response = response.choices[0].message.content

        output['strategy'] = strategy_name
        output['snippet_id'] = snippet_ID
        output['raw_response'] = raw_response
        output['parsed_extraction'] = parse_response(raw_response)

        output['cost_usd'] = compute_cost_usd(
            RATES,
            prompt_tokens=response.usage.prompt_tokens,
            completion_tokens=response.usage.completion_tokens
        )

        output['latency_s'] = elapsed

        return output